# Week 6: Language Models
AI for Good - Minor ADSAI

[Open this notebook in Google Colab](https://colab.research.google.com/github/BTuyn/AI4G/blob/main/Week%206/Week6_Workshop_Student.ipynb) - nothing to install, everything
runs in your browser. If you opened it in Colab already: great.

Today you meet the three ideas behind every modern language model, one at a
time, each one fixing a problem of the one before:

| Step | Model | What it adds |
|------|-------|--------------|
| 1 | **Word embeddings** | words become numbers that carry meaning |
| 2 | **RNN**, then **LSTM** | reading in order, and remembering what you read |
| 3 | **Transformer** | looking at every word at once, so context decides the meaning |

You do **not** build or train any of these today. You use ready-made models
and look at what they do well, where they break, and who gets hurt when they
break. The spine of the week is the same as in Weeks 4 and 5:
**text -> numbers -> model -> prediction**.

Today runs in **three learning waves**, each starting with a short
mini-lecture. A wave starts at every horizontal line in this notebook,
marked like this:

**LEARNING WAVE 1 · SLIDES 4-8**

- Wave 1 - section 1: words as numbers (embeddings)
- Wave 2 - section 2: order and memory (RNN and LSTM)
- Wave 3 - section 3: context and attention (transformers)
- Then section 4: the big individual assignment, and section 5: wrap-up

Do not run ahead into the next wave - wait for the mini-lecture.

### 0. Setup - run these two cells once

Nothing to write here. Run each cell with Shift + Enter and wait until it
finishes (the first one downloads a 66 MB file, a few seconds). If a cell
prints a long warning about a missing token or about "unauthenticated
requests", ignore it.

In [ ]:
# Run, do not read: downloads the pre-trained word embeddings (GloVe, trained by Stanford)
import urllib.request, gzip
import numpy as np
import matplotlib.pyplot as plt

url = "https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-50/glove-wiki-gigaword-50.gz"
urllib.request.urlretrieve(url, "glove50.gz")

words = []
vectors = []
with gzip.open("glove50.gz", "rt", encoding="utf-8") as f:
    f.readline()                       # first line is just file info
    for i, line in enumerate(f):
        if i >= 50000:                 # keep the 50,000 most common words
            break
        parts = line.rstrip().split(" ")
        words.append(parts[0])
        vectors.append(np.array(parts[1:], dtype="float32"))

V = np.vstack(vectors)
V = V / np.linalg.norm(V, axis=1, keepdims=True)   # length 1, so a dot product = similarity
index = {}
for i, w in enumerate(words):
    index[w] = i
print("Loaded", len(words), "words, each one a list of", V.shape[1], "numbers")

In [ ]:
# Run, do not read: small helper functions that we use all day
def vector(word):
    return V[index[word]]

def similarity(a, b):
    # 1.0 = used in exactly the same way, 0 = nothing in common
    return round(float(vector(a) @ vector(b)), 2)

def neighbours(word, n=8):
    scores = V @ vector(word)
    best = np.argsort(-scores)[1:n + 1]
    return [(words[i], round(float(scores[i]), 2)) for i in best]

def analogy(a, b, c, n=5):
    # "a is to b as c is to ?"   (for example: man is to king as woman is to ?)
    target = vector(b) - vector(a) + vector(c)
    scores = V @ target
    result = []
    for i in np.argsort(-scores):
        if words[i] not in (a, b, c):
            result.append(words[i])
        if len(result) == n:
            break
    return result

def plot_words(word_list):
    # squash the 50 numbers per word into 2, so we can draw a map
    X = np.array([vector(w) for w in word_list])
    X = X - X.mean(axis=0)
    u, s, vt = np.linalg.svd(X, full_matrices=False)
    xy = X @ vt[:2].T
    plt.figure(figsize=(8, 6))
    plt.scatter(xy[:, 0], xy[:, 1])
    for k, w in enumerate(word_list):
        plt.text(xy[k, 0] + 0.02, xy[k, 1] + 0.02, w, fontsize=11)
    plt.title("Words on a map (50 numbers squashed into 2)")
    plt.axis("off")
    plt.show()

---

**LEARNING WAVE 1 · SLIDES 4-8**

## 1. Words as numbers

Sections 1.1 to 1.7. From counting words to giving every word a position on a map of meaning.

### 1.1 What is an embedding?

In slide 5 a sentence became a row of word counts. A count tells you *which*
words appear, but not what they *mean*: to a count, `dog` and `cat` are as
unrelated as `dog` and `car`.

An **embedding** fixes that. Every word gets a list of numbers, and words
that are used in similar ways get similar numbers. The file you just loaded
was trained by Stanford on billions of words. It is **pre-trained**: you do
not train anything today.

Run the cell below. It shows the list of numbers for the word `clinic`.

In [ ]:
print(vector("clinic")[:10])
print("This word is a list of", len(vector("clinic")), "numbers. We only printed the first 10.")

Your answer:


clinic is a list of 50 numbers. A single number means nothing on its own; the meaning is in the whole pattern, compared to other words. The numbers are small (around -0.2 to 0.2) because every vector was scaled to length 1 in setup, so comparing two words is just a dot product.

### 1.2 How close are two words?

`similarity(a, b)` compares the two lists of numbers. It gives **1.0** when
two words are used in exactly the same way and values near **0** when they
have nothing in common.

**Predict first.** Which of these five pairs will be the most similar, and
which the least?

- `dog` and `cat`
- `dog` and `car`
- `doctor` and `nurse`
- `clinic` and `hospital`
- `king` and `banana`

Your prediction:


In [ ]:
pairs = [("dog", "cat"), ("dog", "car"), ("doctor", "nurse"), ("clinic", "hospital"), ("king", "banana")]
for pair in pairs:
    print(pair[0], "-", pair[1], ":", similarity(pair[0], pair[1]))

Your answer:


dog-cat scores highest (around 0.9): pets used in the same sentences. doctor-nurse and clinic-hospital are also high: same medical context. dog-car is much lower even though the words look alike, because they’re used differently. king-banana is near 0: nothing in common. Similarity comes from how words are used, not how they’re spelled.

### 1.3 The neighbours of a word

`neighbours("clinic")` lists the words closest to `clinic`. Run it, then try
**three words of your own** (lowercase, and a common word, otherwise it may
not be in the list).

In [ ]:
print(neighbours("clinic"))

In [ ]:
print(neighbours("football"))
print(neighbours("apple"))
print(neighbours("refugee"))

Your answer: The neighbours of clinic are medical words like hospital, clinics, medical and patients. My own words work the same way: football gets other sports and clubs, refugee gets words like camps, asylum and displaced. apple is interesting: it can get fruit words or tech words (microsoft, computer), because one word gets only one set of neighbours. Neighbours are words used in the same contexts, not synonyms.


### 1.4 Play Contexto (10 minutes, in the browser)

Open <https://contexto.me/en/> and, together with your neighbour, guess the
word of the day. After every guess the game tells you how *close* your word
is to the secret word. That is exactly the distance you computed above.

Then answer: how did you decide on your next guess?

Your answer:


### 1.5 A map of words

We cannot draw 50 numbers, so the cell below squashes them into 2 and draws a
map. Run it. Which words sit together? Which word is the odd one out?

In [ ]:
plot_words(["dog", "cat", "horse", "cow", "car", "bus", "train", "bike",
            "doctor", "nurse", "hospital", "clinic", "school", "teacher", "student",
            "king", "queen", "prince", "bread", "rice", "apple", "banana"])

Your answer: The words form clear groups: royalty at the top (king, prince, queen), food on the left (rice, bread, banana, apple), animals in the middle (cow, cat, dog, horse), transport at the bottom (car, bus, train, bike), and people and places in health and education on the right (doctor, nurse, teacher, school, hospital, clinic).


### 1.6 Arithmetic with meaning

Because words are numbers, you can calculate with them.
`analogy("man", "king", "woman")` answers: *man is to king as woman is to ...?*

Run the first two. Then **predict** what `analogy("man", "doctor", "woman")`
will answer, and run it.

In [ ]:
print(analogy("man", "king", "woman"))
print(analogy("paris", "france", "berlin"))

Your prediction: I expect doctor for a woman too, but the model might say nurse, because it learned from text where doctors are mostly men.


In [ ]:
print(analogy("man", "doctor", "woman"))
print(analogy("man", "manager", "woman"))

Your answer: The first two work: queen and germany. Embeddings capture relations like male to female and capital to country. But man : doctor = woman : ? gives words like nurse instead of doctor, and manager gives lower-status jobs. The model isn’t neutral: it copies gender stereotypes from the text it was trained on. If an AI system used this to screen CVs, women could be matched to lower-status jobs. That’s who gets hurt.


### 1.7 One word, one list of numbers

Run the cell below. `bank` has **one** list of numbers, so it has one set of
neighbours.

In [ ]:
print(neighbours("bank"))

Your answer: All neighbours of bank are about money (banking, credit, lender). The other meaning, a river bank, is completely missing. That’s because bank gets one list of numbers, whatever the sentence. The money meaning is much more common in the training text, so it wins. An embedding can’t use context: “I sat on the river bank” and “I put money in the bank” get the same vector. That’s the problem the next waves solve (RNN/LSTM read in order, transformers use context).


---

**LEARNING WAVE 2 · SLIDES 9-14**

## 2. Order and memory: RNN and LSTM

Sections 2.1 to 2.7. Word counts throw away the order of the words. A recurrent network reads the words one by one and carries a memory along. How long can it keep that memory?

### 2.1 Order matters

Bag of words and plain embeddings treat a text as a pile of words. Run the
cell below: it compares the word counts of two sentences with opposite
meanings.

In [ ]:
from collections import Counter

good = "the food was good not bad at all"
bad = "the food was bad not good at all"

print(Counter(good.split()))
print(Counter(bad.split()))
print("Same counts?", Counter(good.split()) == Counter(bad.split()))

Your answer: Both sentences have exactly the same word counts (True), but opposite meanings: one is positive, the other negative. A bag of words throws away word order, so it can’t tell them apart. Meaning lives in the order: “not bad” and “not good” mean opposite things. A model needs to read words in sequence, which is what an RNN does.


### 2.2 Telephone game (in class, 6 minutes, no laptop)

Your teacher will explain the rules. You play two rounds:

1. **The plain chain (the RNN).** A message is whispered down the line. Everyone only knows what the person before them whispered.
2. **The note-taking chain (the LSTM).** Everyone gets a small notebook and may write down one important thing from the message, keep it, and pass it on next to the whisper.

Write down what you notice: what got lost in round 1, and what did the
notebook change in round 2?

Your answer:


### 2.3 The memory test

A tiny game to measure memory. Every **sequence** starts with a secret
colour word (red, green, blue or yellow), followed by a number of filler
words. The model reads the whole sequence, word by word, and then has to say
**which colour the sequence started with**.

The only way to win is to remember the first word until the end.

The cell below prepares two tiny models: an **RNN** and an **LSTM**. It also
contains the function `memory_test(kind, length)`, which trains a model for
a few seconds and returns its score on new sequences (1.0 = always right,
0.25 = just guessing). Run it, then look at two examples.

In [ ]:
# Run, do not read: two tiny models and a game to test their memory
import torch
import torch.nn as nn

colours = ["red", "green", "blue", "yellow"]
fillers = ["the", "cat", "sat", "on", "a", "mat", "and", "looked", "at", "dog"]

def make_data(n, length, generator):
    first = torch.randint(0, 4, (n,), generator=generator)
    filler = torch.randint(4, 14, (n, length), generator=generator)
    return torch.cat([first.unsqueeze(1), filler], dim=1), first

class TinyReader(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.embedding = nn.Embedding(14, 16)
        if kind == "rnn":
            self.reader = nn.RNN(16, 32, batch_first=True)
        else:
            self.reader = nn.LSTM(16, 32, batch_first=True)
            for name, p in self.reader.named_parameters():
                if "bias" in name:
                    p.data[32:64] = 1.0     # start with 'keep what you remember'
        self.answer = nn.Linear(32, 4)

    def forward(self, x):
        out, _ = self.reader(self.embedding(x))
        return self.answer(out[:, -1, :])    # the answer comes after the LAST word

def memory_test(kind, length, seed=1):
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)
    x_train, y_train = make_data(2000, length, g)
    x_test, y_test = make_data(1000, length, g)
    net = TinyReader(kind)
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
    loss_function = nn.CrossEntropyLoss()
    for epoch in range(30):
        order = torch.randperm(2000)
        for i in range(0, 2000, 100):
            batch = order[i:i + 100]
            optimizer.zero_grad()
            loss_function(net(x_train[batch]), y_train[batch]).backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
    with torch.no_grad():
        return round((net(x_test).argmax(1) == y_test).float().mean().item(), 2)

def show_example(length):
    g = torch.Generator().manual_seed(length)
    x, y = make_data(1, length, g)
    tokens = x[0].tolist()
    print(colours[tokens[0]], " ".join(fillers[t - 4] for t in tokens[1:]))
    print("-> the model must answer:", colours[int(y[0])])
    print()

In [ ]:
show_example(5)
show_example(15)

**Predict first.** We test sequences with 5, 10, 20 and 30 filler words
after the colour. For each length, will the model still remember the
colour? Write your guess for the RNN and for the LSTM.

Your prediction: At 5 words both models remember the colour. The RNN gets worse quickly and drops to guessing (0.25) at 20 or 30 words, because each new word overwrites part of its memory. The LSTM stays close to 1.0 at all lengths, because its memory cell can store the colour and ignore the filler words.


### 2.4 Test the RNN

This trains a fresh RNN for each length (about 5 seconds each) and prints
the score.

In [ ]:
lengths = [5, 10, 20, 30]
rnn_scores = []
for length in lengths:
    score = memory_test("rnn", length)
    rnn_scores.append(score)
    print("RNN, after", length, "filler words ->", score)

### 2.5 Test the LSTM

Exactly the same game, with the LSTM instead.

In [ ]:
lstm_scores = []
for length in lengths:
    score = memory_test("lstm", length)
    lstm_scores.append(score)
    print("LSTM, after", length, "filler words ->", score)

### 2.6 Compare

Run the cell below to draw both results in one picture, then answer.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(lengths, rnn_scores, marker="o", label="RNN")
plt.plot(lengths, lstm_scores, marker="o", label="LSTM")
plt.axhline(0.25, color="grey", linestyle="--", label="just guessing")
plt.xlabel("number of words between the secret word and the answer")
plt.ylabel("score (1.0 = always right)")
plt.legend()
plt.show()

Your answer: Up to 10 words both models score 1.0. After that the RNN collapses: 0.5 at 20 words and 0.25 at 30, which is pure guessing. It forgets the first word because every new word overwrites part of its memory (vanishing gradient). The LSTM stays at 1.0 at all lengths: its gates let it store the colour in its memory cell and ignore the filler words, like the notebook in the telephone game. For real texts this matters: in a long sentence the important word (for example “not”) can be far from the end


### 2.7 Your turn: push the LSTM

Test the LSTM on a much longer sequence, for example 50 filler words. Use
`memory_test("lstm", 50)` and print the result. Do it a second time with
`seed=2` as an extra argument (`memory_test("lstm", 50, seed=2)`).

In [20]:
print(memory_test("lstm", 50))
print(memory_test("lstm", 50, seed=2))

0.49
0.76


Your answer: At 50 filler words the LSTM also starts to fail: 0.49 with seed 1 and 0.76 with seed 2. That’s better than guessing (0.25), but far from 1.0. Its memory is longer than the RNN’s, but not unlimited. The different seeds also show that the result depends on luck in training (random start values and data), so one run is not enough to trust a score. This is the limit transformers fix: they don’t pass memory along word by word, they look at every word at once.


---

**LEARNING WAVE 3 · SLIDES 15-20**

## 3. Context and attention: transformers

Sections 3.1 to 3.7. A transformer does not read one word at a time: every word looks at every other word at once. We use two small pre-trained transformers.

### 3.0 Load the transformers

Run this cell once. It downloads two small pre-trained models:
**DistilBERT**, which has been taught to read English (and, in a second
version, to judge whether a text is positive or negative), and **DistilGPT-2**, which has been taught to predict the next
word. This takes up to a minute. Ignore warnings about tokens.

In [ ]:
# Run, do not read: loads two small pre-trained transformers
from transformers import pipeline, AutoModel, AutoModelForCausalLM, AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
sentiment = pipeline("sentiment-analysis", model=checkpoint)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encoder = AutoModel.from_pretrained("distilbert-base-uncased").eval()

lm_tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
lm = AutoModelForCausalLM.from_pretrained("distilgpt2").eval()

def show(sentences):
    for result, sentence in zip(sentiment(sentences), sentences):
        print(result["label"], round(result["score"], 2), "|", sentence)

def word_vector(sentence, word):
    # the list of numbers for ONE word, as this model sees it IN THIS sentence
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = encoder(**enc).last_hidden_state[0]
    position = enc["input_ids"][0].tolist().index(tokenizer.convert_tokens_to_ids(word))
    v = hidden[position]
    return v / v.norm()

def context_similarity(sentence_1, sentence_2, word="bank"):
    return round(float(word_vector(sentence_1, word) @ word_vector(sentence_2, word)), 2)

def next_words(prompt, n=6):
    ids = lm_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = lm(**ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, n)
    for p, i in zip(top.values, top.indices):
        print(round(float(p), 3), "|", lm_tokenizer.decode(int(i)))

def write(prompt, temperature=1.0, length=25, seed=1):
    torch.manual_seed(seed)
    ids = lm_tokenizer(prompt, return_tensors="pt")
    out = lm.generate(**ids, max_new_tokens=length, do_sample=True,
                      temperature=temperature, pad_token_id=lm_tokenizer.eos_token_id)
    print(lm_tokenizer.decode(out[0]).replace("\n", " "))

def pronoun_odds(prompt):
    ids = lm_tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        probs = torch.softmax(lm(**ids).logits[0, -1], dim=-1)
    she = float(probs[lm_tokenizer.encode(" she")[0]])
    he = float(probs[lm_tokenizer.encode(" he")[0]])
    print(prompt, "| she:", round(she, 3), "| he:", round(he, 3))

### 3.1 One word, many meanings

In 1.7 `bank` had one list of numbers, always. A transformer gives a word a
**different list of numbers in every sentence**, because it looks at the
other words first.

`context_similarity(sentence_1, sentence_2)` compares the lists of numbers
of the word `bank` in two sentences. **Predict first:** which comparison
will be the *highest* and which the *lowest*?

- A: `she sat on the bank of the river`
- B: `he deposited money at the bank`
- C: `the bank approved the loan`
- D: `we had a picnic on the river bank`

Your prediction:


In [ ]:
A = "she sat on the bank of the river"
B = "he deposited money at the bank"
C = "the bank approved the loan"
D = "we had a picnic on the river bank"

print("A and D (both river):", context_similarity(A, D))
print("B and C (both money):", context_similarity(B, C))
print("A and B (river / money):", context_similarity(A, B))
print("D and C (river / money):", context_similarity(D, C))

Your answer:


### 3.2 Be attention (in class, 6 minutes)

Your teacher shows a sentence on screen: *The animal didn't cross the street
because it was too tired.* Every student is one word. The student who is `it`
points at the words they need to look at to know what `it` means, and the
rest of the class checks.

Then the teacher changes the end of the sentence to *... because it was too
wide.* Who does `it` look at now?

Write down: which words did `it` look at, and what changed?

Your answer:


### 3.3 Sentiment: where does it fail?

`show([...])` gives you the model's verdict (POSITIVE or NEGATIVE) and how
sure it is (0 to 1) for a list of sentences. Look at the sentences below and
**predict** which ones the model will get wrong.

Your prediction:


In [ ]:
show(["The clinic helped me a lot.",
      "The clinic did not help me at all.",
      "I wouldn't say the staff were unfriendly.",
      "Great, another two-hour wait. Just what I needed.",
      "The doctor was kind, but the waiting room was awful.",
      "the cluic wuz rly good",
      "Het was een heel goede ervaring."])

Your answer:


Now try **three sentences of your own** that you think will fool the
model. Use `show(["sentence one", "sentence two", "sentence three"])`.

Your answer:


### 3.4 What an LLM does: guess the next word

ChatGPT, Claude and Gemini are transformers trained to do one thing:
**predict the next word**. `next_words(prompt)` shows the six most likely
next words (with their probability) according to DistilGPT-2, a small
relative. Run it, then try your own prompt.

In [ ]:
next_words("The doctor said that")

Your answer:


### 3.5 Temperature

`write(prompt, temperature=...)` lets the model keep adding words. The
**temperature** sets how adventurous it is: low = always take the most likely
next word, high = sometimes pick an unlikely one. Run both and compare.

In [ ]:
write("The doctor said that", temperature=0.3)
print()
write("The doctor said that", temperature=1.5)

Your answer:


### 3.6 What did it learn from the internet?

A language model learned from texts written by people. The cell below asks
the model how likely the next word is **she** or **he** after four different
jobs.

In [ ]:
pronoun_odds("The nurse said that")
pronoun_odds("The engineer said that")
pronoun_odds("The doctor said that")
pronoun_odds("The teacher said that")

Your answer:


### 3.7 Who is misread?

You saw three kinds of failure today: a stereotype in the embedding (1.6),
a sarcastic sentence (3.3) and a Dutch sentence (3.3). Imagine a clinic's
helpline that uses a language model to sort incoming messages into *urgent*
and *not urgent*.

Which people would be misread most often, and what would a responsible
clinic do about it?

Your answer:


---

## 4. Big individual assignment: stress-test a language model

**You are a tester, not a builder.** An organisation wants to use the
sentiment model from section 3 to read feedback from its users. Before it
does, it asks you: **where does this model fail, and who would that hurt?**

Pick a context you care about (a clinic, a food bank, a housing association,
a school, a municipality ...). You work with the pre-trained model `sentiment`
and the `show` function from section 3.

**Required skills and outputs**

1. **A test set of at least 12 sentences** about your context, covering at
   least four of these kinds: negation, mixed feelings, sarcasm or irony,
   a long sentence where the key word is far from the end, typos or slang,
   a sentence in another language (for example Dutch).
2. **Your own label first.** Before running anything, write down for each
   sentence what a person would say (positive or negative).
3. **Run the model** and collect the results in a table (a pandas
   DataFrame): sentence, your label, the model's label, its confidence, and
   whether they agree.
4. **Error analysis.** Which kinds of sentences fail? For three failures, say
   what a reader would have needed to get it right (the order of the words,
   a memory of an earlier word, the context around a word) and which step of
   today's ladder (embedding, RNN, LSTM, transformer) would or would not
   have handled it, and why.
5. **A bias probe.** Use `next_words` or `pronoun_odds` with at least two
   prompts that differ in one word (for example two jobs, two names or two
   nationalities) and describe what changes.
6. **A written judgement** (about 150 words): would you let this model read
   the users' feedback? Under which conditions? Who is misread, and what
   does the organisation do about them?

**Bonus options**

- Run your test set through a second model from <https://huggingface.co/models>
  (search for a multilingual sentiment model) by creating a second
  `pipeline("sentiment-analysis", model="...")`, and compare the table.
- Try three different temperatures with `write` for one story prompt and
  describe how the text changes.
- Find a biased analogy in the embeddings of section 1 with `analogy`.

You are graded on the honesty of your error analysis and on your reasoning
about harm, not on how many sentences the model gets right.

In [ ]:
# 1. Context I chose:


# 2. My test sentences and my own labels (write them BEFORE running the model)


# 3. Run the model, build the table (pandas DataFrame)


# 4. Which sentences failed? Why? Which rung of the ladder would have handled them?


# 5. Bias probe (two prompts that differ in one word)


# 6. Written judgement, as a comment or a markdown cell below


Your answer:


---

## 5. Wrap-up

### 5.1 The ladder

Each step fixed a weakness of the one before:

| Model | What it does well | What it still cannot do |
|-------|-------------------|-------------------------|
| Bag of words | counts which words occur, simple and fast | no order, no meaning |
| **Word embeddings** | words that are used alike get similar numbers | one fixed meaning per word, no order |
| **RNN** | reads in order, any length | forgets the beginning of long texts |
| **LSTM** | keeps what matters in a memory lane | reads one word at a time, slow, long texts still hard |
| **Transformer** | every word looks at every other word at once, meaning depends on the context | needs huge data and computers; copies the bias of its data |

### 5.2 Five things to remember

1. A computer does not read. It turns text into numbers and calculates.
2. Embeddings give every word a place on a map of meaning, so similar words end up close.
3. Order matters. An RNN reads in order but forgets; an LSTM remembers longer.
4. A transformer lets every word look at every other word. That is why context finally works, and it is the foundation of ChatGPT, Claude and Gemini.
5. All of these models copy the data they learned from. They sound sure even when they are wrong, so keep a human in the loop and ask who is misread.

### 5.3 Homework (while you work on the hackathon)

Complete these DataCamp chapters:

1. *Recurrent Neural Networks (RNN) for Language Modeling with Keras*: **Chapter 1** and **Chapter 4**
   <https://app.datacamp.com/learn/courses/recurrent-neural-networks-rnn-for-language-modeling-with-keras>
2. *Introduction to LLMs in Python*: the whole chapter **Getting started with Large Language Models (LLMs)**
   <https://campus.datacamp.com/courses/introduction-to-llms-in-python/getting-started-with-large-language-models-llms>

Also finish section 4 and upload the notebook to `Term 1/Week 6/homework/` in your portfolio.

**Next week:** Computer Vision
**Friday:** the hackathon. You find out on the day which tool you use and which problem you solve.